# Submission cross-comparison

Loads every `submissions/*.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

SUB_DIR = Path('submissions')
files = sorted(SUB_DIR.glob('*.csv'))
print('Found submission files:')
for f in files:
    print(' ', f.name)

subs = {}
for f in files:
    df = pd.read_csv(f).sort_values('Id').reset_index(drop=True)
    subs[f.stem] = df

ids_ref = list(subs.values())[0]['Id'].values
for name, df in subs.items():
    assert np.array_equal(df['Id'].values, ids_ref), f'{name} has different Ids!'
print(f'\n{len(subs)} submissions, {len(ids_ref)} test rows each — Ids match.')


## Class distribution per model

In [ ]:
names = list(subs.keys())
n = len(names)
counts = pd.DataFrame(
    {name: subs[name]['Category'].value_counts().sort_index() for name in names}
)
print(counts.T.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(10)
w = 0.8 / n
for i, name in enumerate(names):
    ax.bar(x + i * w, counts[name], width=w, label=name)
ax.set_xticks(x + w * (n - 1) / 2)
ax.set_xticklabels(range(10))
ax.set_xlabel('Digit class')
ax.set_ylabel('# test predictions')
ax.set_title('Predicted class distribution — all models')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## Rows where all models disagree

In [ ]:
# Visualize samples where submissions disagree, with per-model predictions on the same image.
# Build a single table of predictions
pred_df = pd.DataFrame({'Id': ids_ref})
for nm in names:
    pred_df[nm] = subs[nm]['Category'].values

# Keep only disagreement rows
disagreed_mask = pred_df[names].nunique(axis=1) > 1
show_df = pred_df.loc[disagreed_mask].copy()

print(f"Disagreement rows: {len(show_df)}")
if len(show_df) == 0:
    print("No disagreements to visualize.")
else:
    # How many rows to display
    n_show = min(8, len(show_df))
    show_df = show_df.head(n_show).reset_index(drop=True)

    # Resolve image path from Id
    candidate_dirs = [
        Path("data/test/test"),
        Path("data/test"),
        img_dir / "test" if "img_dir" in globals() else None,
        img_dir if "img_dir" in globals() else None,
    ]
    candidate_dirs = [d for d in candidate_dirs if d is not None]

    def get_img_path(img_id):
        if "id_to_img" in globals() and img_id in id_to_img:
            return id_to_img[img_id]
        for d in candidate_dirs:
            p = d / f"{img_id}.png"
            if p.exists():
                return p
        return None

    n_items = len(show_df)
    ncols = len(names) + 1  # image + one panel per submission
    fig, axes = plt.subplots(n_items, ncols, figsize=(2.3 * ncols, 2.4 * n_items), squeeze=False)

    for i, r in show_df.iterrows():
        img_id = int(r["Id"])
        preds = r[names].astype(int)
        majority_pred = int(preds.mode().iloc[0])  # for highlighting outliers
        p = get_img_path(img_id)

        img = None
        if p is not None:
            img = plt.imread(p)

        # First column: base image + Id
        ax0 = axes[i, 0]
        ax0.axis("off")
        if img is None:
            ax0.text(0.5, 0.5, f"Missing image\nId={img_id}", ha="center", va="center")
        else:
            ax0.imshow(img, cmap="gray")
            ax0.set_title(f"Id {img_id}", fontsize=9)

        # One column per submission/model
        for j, nm in enumerate(names, start=1):
            ax = axes[i, j]
            ax.axis("off")

            if img is not None:
                ax.imshow(img, cmap="gray")

            pred = int(r[nm])
            color = "crimson" if pred != majority_pred else "black"
            ax.set_title(f"{nm}\n{pred}", fontsize=8, color=color)

    plt.suptitle("Disagreement samples", y=0.97)
    plt.tight_layout()
    plt.show()
